In [1]:
import numpy as np
import plotly.graph_objects as go

occ_id = 4000836 #923401431 #912350664 #5000321
c1 = 'RdBu_r'
c2 = 'PiYG_r'


data = np.load(f"/marbec-data/RLS-Australia/malpolon/inputs/australia/env/{occ_id}.npy")
data2 = np.load(f"/marbec-data/RLS-Australia/malpolon/inputs/australia/hum/{occ_id}.npy")

In [2]:
def stretch_func(arr, factor, axis):
    length = arr.shape[axis] * factor
    reps = length // arr.shape[axis]
    repeated = np.repeat(arr, reps, axis=axis)
    return repeated.take(range(repeated.shape[axis]-length, repeated.shape[axis]), axis=axis)

def stretch3d(arr, factor=10):
    arr = stretch_func(arr, factor=factor, axis=0)
    arr = stretch_func(arr, factor=factor, axis=1)
    arr = stretch_func(arr, factor=factor, axis=2)
    return arr

def stretch2d(arr, factor=10):
    arr = stretch_func(arr, factor=factor, axis=0)
    arr = stretch_func(arr, factor=factor, axis=1)
    return arr

In [3]:
data = stretch3d(data, factor=20)
data2 = stretch3d(data2, factor=20)

fig = go.Figure()

In [4]:
# Function to create a surface for a given face

def add_face(x, y, z, values, colorscale=c1):

    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        surfacecolor=values,
        colorscale=colorscale,
        cmin=np.min(data if colorscale==c1 else data2),
        cmax=np.max(data if colorscale==c1 else data2),
    ))

In [ ]:
#################### Cube Env ###########################

N, M, P = data.shape
x = np.arange(N)
y = np.arange(M)
z = np.arange(P)


# Top face (z = P-1)
X, Y = np.meshgrid(x, y)
Z = np.full_like(X, P-1)
add_face(X, Y, Z, data[:, :, -1].T)

# Bottom face (z = 0)
Z = np.full_like(X, 0)
add_face(X, Y, Z, data[:, :, 0].T)

# Left face (x = 0)
Y, Z = np.meshgrid(y, z)
X = np.full_like(Y, 0)
add_face(X, Y, Z, data[0, :, :].T)

# Right face (x = N-1)
X = np.full_like(Y, N-1)
add_face(X, Y, Z, data[-1, :, :].T)

# Back face (y = M-1)
X, Z = np.meshgrid(x, z)
Y = np.full_like(X, M-1)
add_face(X, Y, Z, data[:, -1, :].T)

# Front face (y = 0)
Y = np.full_like(X, 0)
add_face(X, Y, Z, data[:, 0, :].T)

#################### Cube Hum ###########################

N, M, P = data2.shape
x = np.arange(N)
y = np.arange(M)
z = np.arange(P)


# Top face (z = P-1)
X, Y = np.meshgrid(x, y)
Z = np.full_like(X, P-1)
add_face(X, Y, Z + 400, data2[:, :, -1].T, colorscale=c2)

# Bottom face (z = 0)
Z = np.full_like(X, 0)
add_face(X, Y, Z + 400, data2[:, :, 0].T, colorscale=c2)

# Left face (x = 0)
Y, Z = np.meshgrid(y, z)
X = np.full_like(Y, 0)
add_face(X, Y, Z + 400, data2[0, :, :].T, colorscale=c2)

# Right face (x = N-1)
X = np.full_like(Y, N-1)
add_face(X, Y, Z + 400, data2[-1, :, :].T, colorscale=c2)

# Back face (y = M-1)
X, Z = np.meshgrid(x, z)
Y = np.full_like(X, M-1)
add_face(X, Y, Z + 400, data2[:, -1, :].T, colorscale=c2)

# Front face (y = 0)
Y = np.full_like(X, 0)
add_face(X, Y, Z + 400 , data2[:, 0, :].T, colorscale=c2)


#################### Layout ###########################


fig.update_layout(width=800, height=800,
    scene=dict(
        xaxis_title='Latitude',
        yaxis_title='Longitude',
        zaxis_title='Human activity           Environment',
        xaxis=dict(showticklabels=False, showline=False, zeroline=False, showgrid=False, titlefont=dict(size=20)),
        yaxis=dict(showticklabels=False, showline=False, zeroline=False, showgrid=False, titlefont=dict(size=20)),
        zaxis=dict(showticklabels=False, showline=False, zeroline=False, showgrid=False, titlefont=dict(size=20)),
    ),
)

fig.update_traces(showscale=False)

fig.add_scatter3d(x=[N/2,N/2], y=[M/2,M/2], z= [0,680], mode='lines', line_width=10, line_color='black')

fig.show()